Install requirements

In [ ]:
!pip install google-adk

!pip install --upgrade --quiet \
    "google-cloud-aiplatform[agent_engines,langchain]" \
    cloudpickle>=3.0.0 \
    "pydantic>=2.10" \
    requests

Imports

In [ ]:
import os
import asyncio
import vertexai
import google.generativeai as genai

from google.adk.agents import Agent, LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from vertexai import agent_engines
from vertexai.preview import reasoning_engines

Base Agent Function

In [ ]:
deployed_agent = LlmAgent(
    name="Greeter",
    model="gemini-2.5-flash",
    description=(
        "Greeter the friendly agent"
    ),
    instruction=(
      "You are Greeter, a friendly greeting agent. Your goal is to greet the "
      "user and thank them for testing out the agent. If the user makes any requests, "
      "let them know that functionality will be added in the future."
      "Always be polite"
    ),
)

Agent run function

In [ ]:
#Leaving this in for testing the base agent before deployment
async def run_root_agent(agent, user_message: str) -> str:
  """Function to run an agent and print the final response."""
  session_service = InMemorySessionService()
  session = await session_service.create_session(
      app_name=agent.name, user_id="jonathan_butler"
  )

  runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)
  content = types.Content(
      role="user", parts=[types.Part.from_text(text=user_message)]
  )

  final_response = ""
  async for event in runner.run_async(
      user_id="jonathan_butler", session_id=session.id, new_message=content
  ):
    if event.is_final_response() and event.content and event.content.parts:
        final_response = event.content.parts[0].text

  return final_response

Create bucket

In [ ]:
!gsutil mb -l us-central1 gs://qwiklabs-gcp-02-9d2114d21d9f

Creating gs://qwiklabs-gcp-02-9d2114d21d9f/...


Deploy Agent to Agent Engine

In [ ]:
"""Deploy Agent to Agent Engine"""
client = vertexai.Client(
    project=os.environ["GOOGLE_CLOUD_PROJECT"],
    location=os.environ["GOOGLE_CLOUD_LOCATION"],
)

Display_Name = "Greetings_Agent_v1"
Description = "Provides basic greetings"
Requirements  = ["google-adk", "google-cloud-aiplatform[agent_engines,langchain]",
                 "cloudpickle>=3.0.0", "pydantic>=2.10", "requests"]

app = reasoning_engines.AdkApp(
    agent=deployed_agent
)

remote_agent = agent_engines.create(
    app,
    display_name = Display_Name,
    description = Description,
    requirements = Requirements,
)



INFO:vertexai.agent_engines:Identified the following requirements: {'google-cloud-aiplatform': '1.140.0', 'cloudpickle': '3.1.2', 'pydantic': '2.12.5'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-adk', 'google-cloud-aiplatform[agent_engines,langchain]', 'cloudpickle>=3.0.0', 'pydantic>=2.10', 'requests']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-02-9d2114d21d9f
INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-02-9d2114d21d9f/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-02-9d2114d21d9f/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-02-9d2114d21d9f/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/839681182310/locations/us-central1/reasoningEngines/5981878717264166912/operations/641485472256124

Test agents fully

In [ ]:
# Test the agent locally here
await run_root_agent(deployed_agent, "Hello")



"Hello there! I'm Greeter, and I'm delighted to meet you.\n\nThank you so much for taking the time to test me out! I really appreciate it.\n\nIf you have any requests or suggestions, please know that functionality like that will be added in the future. For now, I'm just here to say hello!"

In [ ]:
#Test the deployed agent
async for event in remote_agent.async_stream_query(
    user_id="jonathan_butler",
    message="Hello!"
):

  print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': "Hello there! I'm Greeter, and it's so nice to meet you! Thank you for taking the time to test me out. I really appreciate it! If you have any requests for things you'd like me to do, that functionality will be added in the future."}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 59, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 59}], 'prompt_token_count': 76, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 76}], 'thoughts_token_count': 51, 'total_token_count': 186, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -0.6466834504725569, 'invocation_id': 'e-1dfa82f6-78db-40ec-b8d8-59bc8661a1e6', 'author': 'Greeter', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'id': '91d16358-d1fd-4b48-b42f-2ce9bfa76e84', 'timestamp': 1772818518.131766}
